In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_clinica")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto2404")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/medicos.csv"

In [0]:
medicos_schema = StructType(fields=[
    StructField("id_medico", IntegerType(), True),
    StructField("nombre_completo", StringType(), True),
    StructField("sexo", StringType(), True),
    StructField("id_especialidad", IntegerType(), True),
    StructField("turno", StringType(), True),
    StructField("anos_experiencia", IntegerType(), True)
])

In [0]:
df_medicos_read = spark.read.option('header', True)\
                            .schema(medicos_schema)\
                            .csv(ruta)

In [0]:
df_medicos_ingestion = df_medicos_read.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_medicos_final = df_medicos_ingestion.select(
    "id_medico",
    "nombre_completo",
    "sexo",
    "id_especialidad",
    "turno",
    "anos_experiencia",
    "INGESTION_DATE"
)

In [0]:
df_medicos_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.medicos")